# PitSight — NLP + ML Pipeline
## BMW Customer Sentiment Analysis (2019–2025)

---

**Objective:** Extract aspect-level sentiment from 102,848 customer reviews using NLP, then predict customer satisfaction using ML.

**Architecture:**
- **Table 1** (Sales Summary): 22,145 rows — what sold, where, when
- **Table 2** (Customer Experience): 102,848 rows — how customers felt

**Pipeline:**
1. Load & Explore Data
2. NLP — Aspect-Based Sentiment Extraction (6 aspects)
3. Overall Sentiment Scoring (threshold = 4.0)
4. ML Modeling — Predict Satisfied vs Dissatisfied
5. Narrative Validation
6. Export

## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('Libraries loaded.')

In [ ]:
# Upload BMW_Customer_Table2.csv
from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')
print(f'Years: {sorted(df["Year"].unique())}')
print(f'Models: {df["Model"].nunique()}')
df.head(3)

In [ ]:
# Model classifications
EV_MODELS = {'BMW i3', 'BMW i4', 'BMW iX', 'BMW iX3', 'BMW i8'}
M_MODELS = {'BMW M2', 'BMW M3', 'BMW M4', 'BMW M5', 'BMW M8'}
SPORTY_MODELS = M_MODELS | {'BMW Z4', 'BMW i8'}

df['is_ev'] = df['Model'].isin(EV_MODELS).astype(int)

print(f'EV rows: {df["is_ev"].sum():,} ({df["is_ev"].mean()*100:.1f}%)')
print(f'ICE rows: {(1-df["is_ev"]).sum():,} ({(1-df["is_ev"]).mean()*100:.1f}%)')

## 2. NLP — Aspect-Based Sentiment Extraction

### Methodology
We use a **lexicon-based approach** with 6 aspects. Each aspect has:
- **Keywords**: identify if the review mentions this topic
- **Positive indicators**: words signaling satisfaction
- **Negative indicators**: words signaling dissatisfaction

### Critical Design Rules
1. **No keyword overlap** across aspects — each keyword belongs to exactly one aspect
2. **No indicator overlap** — each sentiment word belongs to exactly one aspect (prevents double-counting)
3. **EV vs ICE split** — fuel keywords for ICE, range/charging keywords for EV. NEVER mixed.
4. **Performance only for sporty models** — a BMW 3 Series buyer isn't reviewing track dynamics
5. **Score 0 = not mentioned** — excluded from the average (silence is not an opinion)

### Scoring Algorithm
```
For each review, for each aspect:
  1. Check if ANY keyword is present → if not, score = 0 (not mentioned)
  2. Count positive indicator matches
  3. Count negative indicator matches
  4. raw = positive_count - negative_count
  5. Map: raw <= -2 → 1 | -1 → 2 | 0 → 3 | 1 → 4 | >= 2 → 5
```

In [ ]:
# ASPECT DEFINITIONS — ZERO OVERLAP

ASPECTS = {
    'maintenance': {
        'keywords': [
            'maintenance', 'repair', 'workshop', 'breakdown', 'service cost',
            'repair bill', 'defect', 'warning light', 'transmission',
            'engine warning', 'unplanned', 'routine', 'annual',
            'checkup', 'intervals', 'oil change', 'brake wear', 'drivetrain'
        ],
        'positive': [
            'reasonable', 'affordable', 'predictable', 'straightforward',
            'no unexpected', 'barely spent', 'easy to follow', 'well within',
            'pleasantly', 'cost-effective', 'zero unplanned', 'no major',
            'practically nonexistent', 'fraction', 'okay',
            'no oil change', 'fewer workshop'
        ],
        'negative': [
            'through the roof', 'expensive', 'piling up', 'stranded',
            'shockingly', 'unbearable', 'regret', 'draining', 'monthly routine',
            'multiple times', 'keep coming back', 'higher than', 'way higher',
            'more time in', 'breakdown', 'major repair', 'every service visit',
            'constant', 'third', 'degradation', 'malfunction', 'error',
            'unusual noise'
        ]
    },

    'comfort': {
        'keywords': [
            'comfort', 'seats', 'cabin', 'noise isolation', 'suspension',
            'interior', 'lumbar', 'bumps', 'supportive', 'spacious',
            'road noise', 'whisper quiet', 'ride comfort', 'ride quality'
        ],
        'positive': [
            'exceptional', 'incredibly', 'top notch', 'beautifully',
            'best I have', 'supportive', 'whisper quiet', 'effortless',
            'exceeded', 'comfortable', 'ample', 'spacious'
        ],
        'negative': [
            'uncomfortable', 'too loud', 'harsh', 'does not match',
            'unacceptable'
        ]
    },

    'fuel_ice': {
        'keywords': [
            'fuel', 'mileage', 'gallon', 'fuel efficiency', 'fuel economy',
            'fuel consumption', 'drinks fuel'
        ],
        'positive': [
            'impressive', 'better than', 'surprisingly good',
            'satisfactory', 'practical'
        ],
        'negative': [
            'much higher', 'way too much', 'weakest point',
            'far below', 'drinks fuel'
        ]
    },

    'range_ev': {
        'keywords': [
            'range', 'charging', 'battery', 'charge', 'fast charger',
            'overnight', 'running costs', 'regenerative'
        ],
        'positive': [
            'easily covers', 'consistent', 'accurate', 'seamless',
            'thing of the past', 'drastically reduced', 'impressively'
        ],
        'negative': [
            'anxiety', 'frustrating', 'falls short',
            'painfully slow', 'drains faster'
        ]
    },

    'technology': {
        'keywords': [
            'infotainment', 'idrive', 'carplay', 'software', 'display',
            'driver assist', 'digital', 'instrument cluster',
            'parking assist', 'head up', 'navigation', 'screen', 'app', 'update'
        ],
        'positive': [
            'intuitive', 'responsive', 'flawlessly', 'stress free',
            'improved', 'years ahead', 'crisp', 'easy to navigate',
            'perfectly', 'game changer', 'excellent'
        ],
        'negative': [
            'freezing', 'disconnects', 'false warnings', 'laggy',
            'unresponsive', 'outdated', 'stopped working', 'confusing',
            'counterintuitive', 'glitches', 'unreliable', 'annoyance',
            'sluggish', 'learning curve'
        ]
    },

    'service_experience': {
        'keywords': [
            'service center', 'dealership', 'service advisor', 'staff',
            'wait time', 'pickup', 'drop off', 'appointment',
            'service team', 'management', 'service process',
            'service experience'
        ],
        'positive': [
            'professional', 'efficient', 'knowledgeable', 'transparent',
            'convenient', 'quick', 'minimal', 'updated', 'promptly',
            'courteously', 'above and beyond', 'easy to book', 'on schedule'
        ],
        'negative': [
            'unreasonably long', 'unhelpful', 'dismissive', 'misquoted',
            'escalate', 'delayed', 'without any update', 'could not diagnose',
            'unprofessional', 'rude', 'nightmare', 'unfixed', 'same problem'
        ]
    },

    'performance': {
        'keywords': [
            'acceleration', 'handling', 'engine sound', 'power delivery',
            'torque', 'exhaust', 'track', 'dynamics', 'beast',
            'razor sharp', 'underpowered', 'instant torque'
        ],
        'positive': [
            'breathtaking', 'amazing', 'razor sharp', 'absolute beast',
            'incredible', 'relentless', 'exhilarating', 'addictive',
            'best in class', 'thrilling'
        ],
        'negative': [
            'underpowered', 'not as sharp', 'does not justify',
            'does not feel'
        ]
    }
}

# VERIFY

all_pos = {}
all_neg = {}
for name, config in ASPECTS.items():
    for word in config['positive']:
        if word in all_pos:
            print(f'OVERLAP in positive: \"{word}\" in {all_pos[word]} AND {name}')
        all_pos[word] = name
    for word in config['negative']:
        if word in all_neg:
            print(f'OVERLAP in negative: \"{word}\" in {all_neg[word]} AND {name}')
        all_neg[word] = name

print(f'Positive indicators: {len(all_pos)} (all unique)')
print(f'Negative indicators: {len(all_neg)} (all unique)')
print('Zero overlap confirmed.')

In [ ]:
# SCORING FUNCTION

def compute_aspect_score(text, config):
    """
    Score a single aspect in a review.
    Returns 0 (not mentioned) or 1-5 (very negative to very positive).
    """
    text_lower = text.lower()

    # Step 1: Is this aspect mentioned?
    if not any(kw.lower() in text_lower for kw in config['keywords']):
        return 0  # Not mentioned — excluded from average

    # Step 2: Count sentiment indicators
    pos = sum(1 for p in config['positive'] if p.lower() in text_lower)
    neg = sum(1 for n in config['negative'] if n.lower() in text_lower)

    # Step 3: Map to 1-5
    raw = pos - neg
    if raw <= -2: return 1
    elif raw == -1: return 2
    elif raw == 0: return 3
    elif raw == 1: return 4
    else: return 5


# APPLY TO ALL 102,848 REVIEWS

print('Computing aspect scores...\n')

# Standard aspects
for aspect in ['maintenance', 'comfort', 'technology', 'service_experience']:
    df[f'{aspect}_score'] = df['Customer_Review'].apply(
        lambda t: compute_aspect_score(t, ASPECTS[aspect]))
    mentioned = (df[f'{aspect}_score'] > 0).sum()
    avg = df[df[f'{aspect}_score'] > 0][f'{aspect}_score'].mean()
    print(f'{aspect:25s}: {mentioned:>7,} mentions ({mentioned/len(df)*100:5.1f}%), avg={avg:.2f}')

# Fuel (ICE) or Range (EV) — NEVER mixed
def score_fuel_range(row):
    if row['is_ev']:
        return compute_aspect_score(row['Customer_Review'], ASPECTS['range_ev'])
    else:
        return compute_aspect_score(row['Customer_Review'], ASPECTS['fuel_ice'])

df['fuel_or_range_score'] = df.apply(score_fuel_range, axis=1)
mentioned = (df['fuel_or_range_score'] > 0).sum()
avg = df[df['fuel_or_range_score'] > 0]['fuel_or_range_score'].mean()
print(f'{"fuel_or_range":25s}: {mentioned:>7,} mentions ({mentioned/len(df)*100:5.1f}%), avg={avg:.2f}')
print('  (ICE → fuel keywords | EV → range/charging keywords)')

# Performance — ONLY for sporty models (M-series, Z4, i8)
df['performance_score'] = df.apply(
    lambda r: compute_aspect_score(r['Customer_Review'], ASPECTS['performance'])
    if r['Model'] in SPORTY_MODELS else 0, axis=1)
mentioned = (df['performance_score'] > 0).sum()
avg = df[df['performance_score'] > 0]['performance_score'].mean() if mentioned > 0 else 0
print(f'{"performance":25s}: {mentioned:>7,} mentions ({mentioned/len(df)*100:5.1f}%), avg={avg:.2f}')
print(f'  (Only scored for: {sorted(SPORTY_MODELS)})')

# Verify: no fuel in EV, no range in ICE
ev_reviews = df[df['is_ev']==1]['Customer_Review']
ice_reviews = df[df['is_ev']==0]['Customer_Review']
fuel_in_ev = ev_reviews.str.contains('fuel|gallon|mileage per gallon', case=False).sum()
range_in_ice = ice_reviews.str.contains('charging station|battery range|range anxiety', case=False).sum()
print(f'\nCross-contamination check:')
print(f'  EV reviews mentioning fuel: {fuel_in_ev}')
print(f'  ICE reviews mentioning charging/range: {range_in_ice}')

## 3. Overall Sentiment Scoring

**How it works:**
- Average all NON-ZERO aspect scores (only aspects the customer mentioned)
- Score 0 means "not mentioned" — excluded from average (silence ≠ opinion)

**Threshold: 4.0**
- Score >= 4.0 → **Satisfied** (customer is genuinely happy)
- Score < 4.0 → **Dissatisfied** (customer has complaints or is lukewarm)

Why 4.0? Because 3.0 means "neutral/mentioned but no clear sentiment." A score of 3.2 is someone saying "it's okay" — that's not satisfaction, that's tolerance. True satisfaction starts at 4+.

In [ ]:
THRESHOLD = 4.0

aspect_cols = ['maintenance_score', 'comfort_score', 'fuel_or_range_score',
               'technology_score', 'service_experience_score', 'performance_score']

def compute_sentiment_score(row):
    scores = [row[c] for c in aspect_cols if row[c] > 0]
    return round(np.mean(scores), 2) if scores else 3.0

df['Sentiment_Score'] = df.apply(compute_sentiment_score, axis=1)
df['Sentiment_Label'] = df['Sentiment_Score'].apply(
    lambda x: 'Satisfied' if x >= THRESHOLD else 'Dissatisfied')

print(f'Threshold: {THRESHOLD}')
print(f'\nDistribution:')
print(df['Sentiment_Label'].value_counts())
print(f'\nBalance: {df["Sentiment_Label"].value_counts(normalize=True).round(3).to_dict()}')
print(f'\nSentiment Score stats:')
print(df['Sentiment_Score'].describe().round(2))

In [ ]:
# Visualize sentiment distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['Sentiment_Score'], bins=40, color='#4361EE', edgecolor='white', alpha=0.8)
axes[0].axvline(x=THRESHOLD, color='#E63946', linestyle='--', linewidth=2, label=f'Threshold = {THRESHOLD}')
axes[0].set_title('Sentiment Score Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Pie
counts = df['Sentiment_Label'].value_counts()
axes[1].pie(counts, labels=counts.index, autopct='%1.1f%%',
           colors=['#2A9D8F', '#E63946'], textprops={'fontsize': 12})
axes[1].set_title('Satisfied vs Dissatisfied', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. ML Modeling — Predict Satisfied vs Dissatisfied

### Objective
Can we predict customer satisfaction using **only operational metrics** — without reading the review?

### Features Used
| Feature | What it measures |
|---------|------------------|
| Service_History | Number of workshop visits |
| Delivery_Days | How long the customer waited |
| Customer_Age | Demographic |
| Warranty_Years | Coverage period |
| is_ev | Electric vs ICE |

### Models Compared
| Model | Why |
|-------|-----|
| Logistic Regression | Linear baseline — shows if patterns are simple |
| Random Forest | Handles non-linear relationships, provides feature importance |
| Gradient Boosting | Sequential error correction, usually top accuracy on tabular data |

### Why NOT
- **XGBoost**: Functionally equivalent to sklearn's GradientBoosting at 100K rows
- **Deep Learning**: Tree models consistently outperform neural nets on tabular data
- **Transformers (BERT)**: Process text, not tabular features. We deliberately separate NLP from ML.

In [ ]:
# DATA PREP

feature_cols = ['Service_History', 'Delivery_Days', 'Customer_Age', 'Warranty_Years', 'is_ev']

X = df[feature_cols]
y = (df['Sentiment_Label'] == 'Satisfied').astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'Satisfied: {y.sum():,} ({y.mean()*100:.1f}%) | Dissatisfied: {(1-y).sum():,} ({(1-y).mean()*100:.1f}%)')

In [ ]:
# TRAIN & COMPARE 3 MODELS

models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=42), True),
    'Random Forest': (RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_split=10,
        min_samples_leaf=5, random_state=42, n_jobs=-1), False),
    'Gradient Boosting': (GradientBoostingClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        subsample=0.8, min_samples_split=10, random_state=42), False),
}

results = {}
for name, (model, use_scaled) in models.items():
    Xtr = X_train_s if use_scaled else X_train
    Xte = X_test_s if use_scaled else X_test

    model.fit(Xtr, y_train)
    pred = model.predict(Xte)
    acc = accuracy_score(y_test, pred)
    cv_s = cross_val_score(model, Xtr, y_train, cv=cv, scoring='accuracy')

    results[name] = {'test': acc, 'cv_mean': cv_s.mean(), 'cv_std': cv_s.std(),
                     'model': model, 'pred': pred}

    print(f'\n{"="*55}')
    print(f'  {name}')
    print(f'{"="*55}')
    print(f'  Test: {acc:.4f} | CV: {cv_s.mean():.4f} +/- {cv_s.std():.4f}')
    print(classification_report(y_test, pred, target_names=['Dissatisfied', 'Satisfied']))

best_name = max(results, key=lambda k: results[k]['cv_mean'])
print(f'\nBest Model: {best_name} (CV: {results[best_name]["cv_mean"]:.4f})')

In [ ]:
# FEATURE IMPORTANCE & CONFUSION MATRIX

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Feature importance
best_model = results[best_name]['model']
if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values()
    imp.plot(kind='barh', ax=axes[0], color='#4361EE', edgecolor='white')
    axes[0].set_title(f'Feature Importance ({best_name})', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Importance')

# Confusion matrix
cm = confusion_matrix(y_test, results[best_name]['pred'])
ConfusionMatrixDisplay(cm, display_labels=['Dissatisfied', 'Satisfied']).plot(ax=axes[1], cmap='Blues')
axes[1].set_title(f'Confusion Matrix ({best_name})', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Print importance values
if hasattr(best_model, 'feature_importances_'):
    imp_sorted = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
    print('\nFeature Importance:')
    for f, v in imp_sorted.items():
        print(f'  {f:20s}: {v:.4f} ({v*100:.1f}%)')

## 5. Narrative Validation

Do the NLP-derived sentiments match the sales trends in Table 1?

In [ ]:
# X6 DECLINE | X5 RECOVERY

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

for col_idx, (model, color, title) in enumerate([
    ('BMW X6', '#E63946', 'BMW X6 — Decline'),
    ('BMW X5', '#2A9D8F', 'BMW X5 — Recovery')
]):
    subset = df[df['Model'] == model]

    # Service History trend
    svc = subset.groupby('Year')['Service_History'].mean()
    axes[0, col_idx].plot(svc.index, svc.values, marker='o', linewidth=2.5, color=color)
    axes[0, col_idx].set_title(f'{title} — Avg Service Visits', fontsize=13, fontweight='bold')
    axes[0, col_idx].set_ylabel('Avg Service History')

    # Satisfaction %
    sat = subset.groupby('Year')['Sentiment_Label'].apply(lambda x: (x=='Satisfied').mean() * 100)
    axes[1, col_idx].bar(sat.index, sat.values, color=color, alpha=0.7, edgecolor='white')
    axes[1, col_idx].set_title(f'{title} — % Satisfied', fontsize=13, fontweight='bold')
    axes[1, col_idx].set_ylabel('% Satisfied')
    axes[1, col_idx].set_ylim(0, 105)

plt.tight_layout()
plt.show()

# Print numbers
for model_name in ['BMW X6', 'BMW X5']:
    print(f'\n{model_name}:')
    sub = df[df['Model']==model_name]
    for yr in range(2019, 2026):
        s = sub[sub['Year']==yr]
        if len(s) > 0:
            print(f'  {yr}: Svc={s["Service_History"].mean():.1f} | {(s["Sentiment_Label"]=="Satisfied").mean()*100:.0f}% Satisfied | Maint={s[s["maintenance_score"]>0]["maintenance_score"].mean():.1f}')

In [ ]:
# M-SERIES YOUTH SHIFT | EV ADOPTION

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# M-Series age shift
m = df[df['Model'].isin(M_MODELS)]
m_age = m.groupby('Year')['Customer_Age'].mean()
m_sat = m.groupby('Year')['Sentiment_Label'].apply(lambda x: (x=='Satisfied').mean()*100)

ax1 = axes[0]
ax2 = ax1.twinx()
ax1.plot(m_age.index, m_age.values, 'o-', color='#E63946', linewidth=2.5, label='Avg Age')
ax2.bar(m_sat.index, m_sat.values, color='#4361EE', alpha=0.3, label='% Satisfied')
ax1.set_title('M-Series: Youth Marketing Shift', fontsize=14, fontweight='bold')
ax1.set_ylabel('Avg Buyer Age', color='#E63946')
ax2.set_ylabel('% Satisfied', color='#4361EE')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')

# EV satisfaction improvement
ev = df[df['is_ev']==1]
ev_svc = ev.groupby('Year')['Service_History'].mean()
ev_sat = ev.groupby('Year')['Sentiment_Label'].apply(lambda x: (x=='Satisfied').mean()*100)

ax3 = axes[1]
ax4 = ax3.twinx()
ax3.plot(ev_svc.index, ev_svc.values, 's-', color='#F77F00', linewidth=2.5, label='Avg Service Visits')
ax4.bar(ev_sat.index, ev_sat.values, color='#2A9D8F', alpha=0.3, label='% Satisfied')
ax3.set_title('EV Models: Improving Satisfaction', fontsize=14, fontweight='bold')
ax3.set_ylabel('Avg Service Visits', color='#F77F00')
ax4.set_ylabel('% Satisfied', color='#2A9D8F')
ax3.legend(loc='upper left')
ax4.legend(loc='upper right')

plt.tight_layout()
plt.show()

## 6. Export

In [ ]:
# Additional features for export
df['review_length'] = df['Customer_Review'].str.len()
df['review_word_count'] = df['Customer_Review'].str.split().str.len()

neg_words = ['not','no','never','terrible','horrible','disappointing','frustrating',
             'expensive','nightmare','unbearable','rude','broke','broken','failed',
             'worst','regret','painful','anxiety','annoying','unfixed','stranded',
             'draining','piling','shockingly','laggy','glitches']
df['negative_word_ratio'] = df['Customer_Review'].apply(
    lambda t: round(sum(1 for w in t.lower().split() if w.strip('.,!?') in neg_words) / max(len(t.split()),1), 4))

output_file = 'BMW_Table2_with_NLP.csv'
df.to_csv(output_file, index=False)

print(f'Saved: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'\nFinal columns:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col}')

from google.colab import files
files.download(output_file)